# 02_params_result_and_box — Params, Result, and box.info

Typed input parameters and output result. Logging via `box.info()` with variable substitution and color.

**What's new** (in addition to example 01):

| Concept | Description |
|---------|-------------|
| `BaseParams` | Base class for typed input parameters |
| `BaseResult` | Base class for typed output result |
| `Field(description=...)` | Pydantic field with a description |
| `box.info(Channel, template, **kwargs)` | Structured aspect logger |
| `Channel.business` | Semantic tag: this is a business event |
| `{%var.key}` | Substitutes a kwargs value into the log template |
| `{%var.key\|cyan}` | Same, but with ANSI color |
| `LogCoordinator + ConsoleLogger` | Where box events are written |

**Why `box.info` and not `print`?**
- `print` writes a string to stdout: no context, no filtering, nothing.
- `box.info(...)` creates a structured event carrying channel, level, domain, aspect name, operation name. Can be filtered on any dimension, routed to multiple sinks, written to an OCEL log.
- Use `print` only in the outer runner for quick demos. Use `box` always inside aspects.

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain

In [ ]:
class GreetingDomain(BaseDomain):
    name = "greeting"
    description = "Greetings domain"

## Params & Result

`BaseParams` and `BaseResult` are Pydantic models. Fields are described with `Field(description=...)`.

The description in `Field` is **not a code comment** — it goes into the OpenAPI schema and MCP tool. Fields are accessible directly inside the aspect: `params.name`.

In [ ]:
class GreetParams(BaseParams):
    name: str = Field(description="Name of the person to greet")


class GreetResult(BaseResult):
    message: str = Field(description="Assembled greeting message")

## Action

`box.info(channel, template, **kwargs)` creates a structured log event:
- **channel** — `Channel.business` marks this as a business event. Channels: `business`, `debug`, `security`, `compliance`, `error`. Combine with `|`: `Channel.business | Channel.security`.
- **template** — `{%var.name}` inserts the value from kwargs. `{%var.name|cyan}` inserts it colored cyan. Available colors: `red`, `green`, `yellow`, `blue`, `cyan`, `magenta`, `white`, `grey` and their `bright_` variants.

By default `ActionProductMachine` doesn't know where to write events — `box.info` is silently ignored. Pass `LogCoordinator` to route events to a sink.

In [ ]:
@meta(description="Greet a person by name", domain=GreetingDomain)
@check_roles(GuestRole)
class GreetPersonAction(BaseAction[GreetParams, GreetResult]):

    @summary_aspect("Build greeting and return result")
    async def greet_summary(self, params, state, box, connections):
        await box.info(
            Channel.business,
            "Greeting: Hello, {%var.name|cyan}!",
            name=params.name,
        )
        return GreetResult(message=f"Hello, {params.name}!")

## Run

`LogCoordinator` is the event bus for all logging. `ConsoleLogger` is a built-in sink that prints to stdout.

If you don't need a logger — just don't pass `log_coordinator`. The `box.info` calls stay in the code and won't break anything: events simply have nowhere to go and are silently discarded.

> In Colab, `await` works directly in cells — no need for `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()]),
    )
    result = await machine.run(
        Context(),
        GreetPersonAction(),
        GreetParams(name="Alice"),
    )
    print(f"\nResult: {result.message}")

await main()